# consoleBert Masked BERT Example
This notebook shows how to use a lightweight BERT masked language model to change the semantics of a specific word in a sentence. It also demonstrates how to choose replacements using semantic similarity.

In [ ]:
# Install libraries
!pip install -q transformers sentence-transformers torch

In [ ]:
from transformers import AutoTokenizer, AutoModelForMaskedLM, pipeline
from sentence_transformers import SentenceTransformer
import numpy as np
import re

## Load a small masked language model

In [ ]:
model_name = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)
fill_mask = pipeline('fill-mask', model=model, tokenizer=tokenizer, top_k=10)
semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

## Helper functions

In [ ]:
def mask_word(text, target_word):
    mask_token = tokenizer.mask_token
    pattern = re.compile(rf'\b{re.escape(target_word)}\b', flags=re.IGNORECASE)
    if not pattern.search(text):
        return None
    return pattern.sub(mask_token, text, count=1)

def pick_best_candidate(candidates, meaning=None):
    if not meaning or len(candidates) == 1:
        return candidates[0] if candidates else None
    target_emb = semantic_model.encode([meaning], convert_to_numpy=True)[0]
    cand_embs = semantic_model.encode(candidates, convert_to_numpy=True)
    scores = np.dot(cand_embs, target_emb) / (np.linalg.norm(cand_embs, axis=1) * np.linalg.norm(target_emb) + 1e-9)
    return candidates[int(np.argmax(scores))] if len(scores) else None

def transform_sentence(text, target_word=None, target_meaning=None):
    if not target_word:
        words = re.findall(r'\b\w+\b', text)
        if not words:
            return text
        target_word = max(words, key=len)
    masked_text = mask_word(text, target_word)
    if not masked_text:
        return text
    outputs = fill_mask(masked_text)
    candidates = [o['token_str'].strip() for o in outputs if o['token_str'].strip().lower() != target_word.lower()]
    if not candidates:
        return text
    best = pick_best_candidate(candidates, target_meaning)
    return masked_text.replace(tokenizer.mask_token, best, 1)

## Example: change the semantics of a word

In [ ]:
text = 'Please send me the draft by Friday.'
target_word = 'draft'
meaning = 'a quick note instead of a long document'
print('Original:', text)
print('Transformed:', transform_sentence(text, target_word, meaning))

## Next steps for training on Colab
- Collect examples of sentences and their desired word-level semantic changes.
- Fine-tune the masked language model on your custom examples using `Trainer` or `TrainerCallback`.
- Save the fine-tuned model and use it in `consoleBert/server/main.py` by changing `MODEL_NAME` to the saved folder.